In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.6 MB/s eta 0:00:00


In [ ]:
import gc
import time
import torch
import transformers
import bitsandbytes as bnb

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PROMPT = "Explain how an LLM generates text, step by step."
MAX_NEW_TOKENS = 64

assert torch.cuda.is_available(), "Enable a GPU runtime first."

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")

print("Prompt tokens:", inputs["input_ids"].shape[1])

PyTorch: 2.11.0+cu130
Transformers: 5.20.0
BitsAndBytes: 0.50.2
GPU: Tesla T4
GPU memory (GiB): 14.56


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Prompt tokens: 13


In [ ]:
def benchmark_model(model, label, load_seconds):
    model.eval()

    # Warmup
    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    torch.cuda.synchronize()

    # Reset peak after warmup
    torch.cuda.reset_peak_memory_stats()

    times = []
    output = None

    for _ in range(3):
        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
            )

        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    generated_tokens = output.shape[1] - inputs["input_ids"].shape[1]

    result = {
        "model": label,
        "load_seconds": round(load_seconds, 2),
        "model_footprint_gib": round(
            model.get_memory_footprint() / 1024**3, 3
        ),
        "peak_gpu_allocated_gib": round(
            torch.cuda.max_memory_allocated() / 1024**3, 3
        ),
        "mean_generation_seconds": round(sum(times) / len(times), 3),
        "generated_tokens": generated_tokens,
        "output_text": tokenizer.decode(
            output[0], skip_special_tokens=True
        ),
    }

    return result

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="sdpa",
).to("cuda")

torch.cuda.synchronize()
fp16_load_seconds = time.perf_counter() - start

fp16_result = benchmark_model(
    model_fp16, "FP16", fp16_load_seconds
)

print(fp16_result)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'model': 'FP16', 'load_seconds': 57.73, 'model_footprint_gib': 2.875, 'peak_gpu_allocated_gib': 2.89, 'mean_generation_seconds': 2.291, 'generated_tokens': 64, 'output_text': "Explain how an LLM generates text, step by step. An LLM (Large Language Model) is a type of artificial intelligence that can generate human-like text based on the input it receives. Here's a step-by-step explanation of how an LLM generates text:\n\n1. Input: The first step in generating text with an LLM is to provide it with some form of"}


In [ ]:
del model_fp16
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

start = time.perf_counter()

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

torch.cuda.synchronize()
int4_load_seconds = time.perf_counter() - start

int4_result = benchmark_model(
    model_4bit, "4-bit NF4", int4_load_seconds
)

print(int4_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'model': '4-bit NF4', 'load_seconds': 12.84, 'model_footprint_gib': 1.045, 'peak_gpu_allocated_gib': 1.089, 'mean_generation_seconds': 5.136, 'generated_tokens': 64, 'output_text': "Explain how an LLM generates text, step by step. An LLM (Language Model) is a type of artificial intelligence that can generate human-like text based on the input it receives. Here's how an LLM typically works to generate text:\n\n1. Input: The user provides input in the form of a prompt or question. This could be anything from a simple sentence to"}


In [ ]:
print("\n=== FP16 vs 4-bit NF4 ===")

print("FP16 model footprint:",
      fp16_result["model_footprint_gib"], "GiB")
print("4-bit model footprint:",
      int4_result["model_footprint_gib"], "GiB")

footprint_reduction = (
    1 - int4_result["model_footprint_gib"]
    / fp16_result["model_footprint_gib"]
) * 100

print("Model footprint reduction:",
      round(footprint_reduction, 1), "%")

print("FP16 peak GPU allocation:",
      fp16_result["peak_gpu_allocated_gib"], "GiB")
print("4-bit peak GPU allocation:",
      int4_result["peak_gpu_allocated_gib"], "GiB")

print("FP16 mean generation latency:",
      fp16_result["mean_generation_seconds"], "s")
print("4-bit mean generation latency:",
      int4_result["mean_generation_seconds"], "s")

print("\nFP16 output:\n", fp16_result["output_text"])
print("\n4-bit output:\n", int4_result["output_text"])


=== FP16 vs 4-bit NF4 ===
FP16 model footprint: 2.875 GiB
4-bit model footprint: 1.045 GiB
Model footprint reduction: 63.7 %
FP16 peak GPU allocation: 2.89 GiB
4-bit peak GPU allocation: 1.089 GiB
FP16 mean generation latency: 2.291 s
4-bit mean generation latency: 5.136 s

FP16 output:
 Explain how an LLM generates text, step by step. An LLM (Large Language Model) is a type of artificial intelligence that can generate human-like text based on the input it receives. Here's a step-by-step explanation of how an LLM generates text:

1. Input: The first step in generating text with an LLM is to provide it with some form of

4-bit output:
 Explain how an LLM generates text, step by step. An LLM (Language Model) is a type of artificial intelligence that can generate human-like text based on the input it receives. Here's how an LLM typically works to generate text:

1. Input: The user provides input in the form of a prompt or question. This could be anything from a simple sentence to
